# Build profiles_v5 (10k halos/bin)

Same pipeline as `profiles_v3.hdf5` (built by `Matched_make_read_s1.ipynb`), with one intentional
change: **`NMAX` 1000 -> 10000** (the per-bin halo cap). Two other fixes are folded in (see the
`save_bin`/`select_halos` cells for exactly what and why):

1. **Resume safety** -- a bin is only considered "done" if it has a `complete=True` attr written
   as the very last step of saving it, not just by checking whether its HDF5 group exists. If the
   process is killed mid-write, that bin gets correctly redone on restart instead of silently
   treated as finished forever. This matters a lot for a run expected to take about a week.
2. **Halo traceability** -- `dmo_halo_id`/`hyd_halo_id` are now saved with every bin (they exist
   in `final_matched.hdf5` but `profiles_v3`'s `select_halos` never read them), so any bin can be
   traced back to the exact halos that went into it.

Output goes to `profiles_v5_10k.hdf5` -- a **new** file, `profiles_v3.hdf5` is never touched.

`run_sweep()` is fully resumable: safe to interrupt (Ctrl-C, crash, reboot) and rerun the same
cell at any time -- it picks back up from the last *actually completed* bin.

In [1]:
import os, time
import numpy as np
import h5py
from scipy.spatial import cKDTree
import Func_for_fit as F

In [2]:
# ==== sims / snapshots / mass bins -- identical to profiles_v3 ====
SIMS = ["fgas+2sigma","fgas-2sigma","fgas-4sigma","fgas-8sigma",
        "Mstar-1sigma","Mstar-1sigma_fgas-4sigma","Jet","Jet_fgas-4sigma"]
SNAPS = [9,13,17,37,47,57,62,67,69,71,73,75,77]
MASS_BINS = [(lo, round(lo+0.125,3)) for lo in np.arange(10.0,15.0,0.125)]   # 40 bins, 0.125 dex

# ==== particle/catalog file layout -- identical to profiles_v3 ====
BOX_MPC    = 1000.0
R_MAX_NORM = 5.0
MATCH_FILE = "final_matched.hdf5"
DMO_DIR    = "L1_m9_DMO"
HYD_DIR    = "flamingo_downsampled/{sim}"
FNAME      = "flamingo_{snap:04d}.hdf5"
_PARTS     = {"DMParticles":"Masses","GasParticles":"Masses",
              "StarsParticles":"Masses","BHParticles":"DynamicalMasses"}

# ==== L1_m9 softening (Schaye+2023 Table 2) -- identical to profiles_v3 ====
EPS_COM_CKPC, EPS_PROP_PKPC = 22.3, 5.70
def eps_eff_ckpc(a):
    """Effective softening in COMOVING kpc: min(e_com, e_prop/a). Switches at a~0.256 (z~2.9)."""
    return min(EPS_COM_CKPC, EPS_PROP_PKPC/a)

# ==== run parameters -- THE ONLY INTENTIONAL CHANGE from profiles_v3 is NMAX 1000 -> 10000 ====
N_LOG   = 99
NMAX    = 10000                      # was 1000 in profiles_v3
SEED    = 0
X10_MAX = 0.3
OUT_H5  = "profiles_v5_10k.hdf5"     # NEW filename -- never overwrites profiles_v3.hdf5

In [3]:
def make_edges(a, r200_ref, n_log=99):
    """Bin0=[0,10*eps], then n_log log bins to 5.0. Units r/R200c. eps comoving (ckpc->cMpc)."""
    x0 = 10.0*eps_eff_ckpc(a)/1000.0/r200_ref
    return np.concatenate([[0.0], np.logspace(np.log10(x0), np.log10(5.0), n_log+1)])

def hist_one(d, m, edges, r200):
    """Raw mass+count hist for one halo; d,m are its own particles (comoving offsets, masses)."""
    nb = len(edges)-1
    if len(m)==0: return np.zeros(nb), np.zeros(nb, np.int64)
    rn = np.sqrt((d*d).sum(1))/r200
    mh,_ = np.histogram(rn, bins=edges, weights=m)
    nh,_ = np.histogram(rn, bins=edges)
    return mh, nh.astype(np.int64)

In [4]:
def read_a(snap, sim):
    """Scale factor from the hydro snapshot Header (single source of truth)."""
    with h5py.File(f"{HYD_DIR.format(sim=sim)}/{FNAME.format(snap=snap)}", "r") as f:
        return float(np.asarray(f["Header"].attrs["Scale-factor"]).ravel()[0])

def select_halos(snap, sim, lo, hi, a, nmax=None, seed=0):
    """Matched halos in [lo,hi) log-mass. Centre+r200 -> comoving (/a). Catalogue only.

    FIX vs profiles_v3: also reads dmo_halo_id/hyd_halo_id (they exist in final_matched.hdf5,
    profiles_v3's select_halos never read them) so every saved bin can be traced back to the
    exact halos that went into it.
    """
    with h5py.File(MATCH_FILE, "r") as f:
        g = f[f"{snap}/{sim}"]; ok = g["matched"][:]>0; m = g["dmo_m200c"][:]
        sel = np.flatnonzero(ok & (m>=10**lo) & (m<10**hi))
        if nmax and len(sel)>nmax:
            sel = np.random.default_rng(seed).choice(sel, nmax, replace=False)
        sel = np.sort(sel)                                   # h5py fancy-index needs increasing order
        return {"dmo_centre":g["dmo_centre"][sel]/a, "dmo_r200":g["dmo_r200c"][sel]/a,
                "hyd_centre":g["hyd_centre"][sel]/a, "hyd_r200":g["hyd_r200c"][sel]/a,
                "dmo_m200":m[sel], "hyd_m200":g["hyd_m200c"][sel],
                "dmo_halo_id":g["dmo_halo_id"][sel], "hyd_halo_id":g["hyd_halo_id"][sel],
                "n":len(sel)}

def _read(path, parts):
    pos, ms = [], []
    with h5py.File(path, "r") as f:
        for p in parts:
            if p not in f or "Coordinates" not in f[p]: continue
            c = f[p]["Coordinates"][:]
            if c.shape[0]==0: continue
            mk = _PARTS[p]
            if mk not in f[p]: raise KeyError(f"{p}/{mk} missing in {path}: has {list(f[p])}")
            pos.append(c); ms.append(f[p][mk][:])
    if not pos: raise ValueError(f"no particles in {path}")
    return np.concatenate(pos)%BOX_MPC, np.concatenate(ms)   # comoving, native dtype

def load_snap_data(snap, sim):
    """All particles (DMO: DM; hydro: DM+gas+stars+BH) + periodic trees, built once per (snap,sim)."""
    dp,dm = _read(f"{DMO_DIR}/{FNAME.format(snap=snap)}", ["DMParticles"])
    hp,hm = _read(f"{HYD_DIR.format(sim=sim)}/{FNAME.format(snap=snap)}", list(_PARTS))
    return {"dmo_pos":dp,"dmo_m":dm,"dmo_tree":cKDTree(dp,boxsize=BOX_MPC),
            "hyd_pos":hp,"hyd_m":hm,"hyd_tree":cKDTree(hp,boxsize=BOX_MPC),
            "a":read_a(snap,sim)}

In [5]:
def get_particles(halos, snap_data, n_min=0):
    """Per-halo (pos,mass) within R_MAX_NORM*r200. DMO side on DMO centre, HYDRO side on HYDRO
    centre (each halo centred at r=0 on its own peak -> shape comparison, location removed)."""
    n = halos["n"]
    out = {"dmo":[],"hyd":[],"n":n,"n_dmo":np.zeros(n,np.int64),"n_hyd":np.zeros(n,np.int64),
           "keep":np.zeros(n,bool),"empty":n==0}
    if n==0: return out
    def one(tree,pos,m,c,r):
        idx = tree.query_ball_point(c%BOX_MPC, R_MAX_NORM*r)
        if not idx: return np.empty((0,3)), np.empty(0)
        d = pos[idx]-c; d -= BOX_MPC*np.round(d/BOX_MPC)
        return d, m[idx]
    for i in range(n):
        cd=halos["dmo_centre"][i].astype(np.float64); rd=float(halos["dmo_r200"][i])
        ch=halos["hyd_centre"][i].astype(np.float64); rh=float(halos["hyd_r200"][i])
        pd,md = one(snap_data["dmo_tree"],snap_data["dmo_pos"],snap_data["dmo_m"],cd,rd)
        ph,mh = one(snap_data["hyd_tree"],snap_data["hyd_pos"],snap_data["hyd_m"],ch,rh)
        out["dmo"].append((pd,md)); out["hyd"].append((ph,mh))
        out["n_dmo"][i]=len(md); out["n_hyd"][i]=len(mh)
    out["keep"] = (out["n_dmo"]>=n_min)&(out["n_hyd"]>=n_min)
    return out

In [6]:
def ot_one_bin(h, p, a, lo, hi, n_log=N_LOG):
    """Histograms+OT for one pre-selected mass bin. Average over all halos with >=1 particle per
    side; flags empty/degenerate bins, doesn't drop them. status: 'ok'|'all_empty'|'degenerate'."""
    N=h["n"]; nb=n_log+1
    out={"lo":lo,"hi":hi,"a":a,"n_halos":N,"status":"ok","ot":None}
    r200_ref=float(np.median(h["dmo_r200"])); out["dmo_r200_med"]=r200_ref
    edges=make_edges(a,r200_ref,n_log); rc=np.sqrt(edges[:-1]*edges[1:]); rc[0]=edges[1]/2.0
    out["edges"]=edges; out["rc"]=rc; out["x10"]=float(edges[1])
    m_dmo=np.zeros((N,nb)); m_hyd=np.zeros((N,nb))
    c_dmo=np.zeros(nb,np.int64); c_hyd=np.zeros(nb,np.int64)
    has_d=np.zeros(N,bool); has_h=np.zeros(N,bool)
    for i in range(N):
        md,nd = hist_one(*p["dmo"][i], edges, float(h["dmo_r200"][i]))
        mh,nh = hist_one(*p["hyd"][i], edges, float(h["hyd_r200"][i]))
        c_dmo+=nd; c_hyd+=nh; sdv,shv = md.sum(), mh.sum()
        if sdv>0: m_dmo[i]=md/sdv; has_d[i]=True
        if shv>0: m_hyd[i]=mh/shv; has_h[i]=True
    out.update(m_dmo=m_dmo,m_hyd=m_hyd,cnt_dmo=c_dmo,cnt_hyd=c_hyd,
        n_empty_dmo=int((c_dmo==0).sum()), n_empty_hyd=int((c_hyd==0).sum()),
        n_used_dmo=int(has_d.sum()), n_used_hyd=int(has_h.sum()),
        has_dmo=has_d, has_hyd=has_h,
        npart_dmo=p["n_dmo"], npart_hyd=p["n_hyd"],
        npart_dmo_med=float(np.median(p["n_dmo"])), npart_hyd_med=float(np.median(p["n_hyd"])))
    if not (has_d.any() and has_h.any()):
        out["status"]="all_empty"; out["avg_dmo"]=out["avg_hyd"]=np.full(nb,np.nan)
    else:
        ad,ah = m_dmo[has_d].mean(0), m_hyd[has_h].mean(0)
        out["avg_dmo"],out["avg_hyd"]=ad,ah
        if ad.sum()<=0 or ah.sum()<=0 or not np.all(np.isfinite(ad+ah)):
            out["status"]="degenerate"
        else:
            try: out["ot"]=F.ot_map_from_hists(edges,ad,ah,rc)
            except Exception as e: out["status"]="degenerate"; out["err"]=str(e)
    out["ot_reliable"]=(out["status"]=="ok" and out["x10"]<X10_MAX
        and out["n_empty_dmo"]==0 and out["n_empty_hyd"]==0
        and min(out["n_used_dmo"],out["n_used_hyd"])>=0.9*N)
    return out

## Smoke test -- verify the pipeline before launching the real sweep

Doesn't touch any output file. Safe to rerun any time, e.g. after editing anything above.

In [7]:
# quick single-bin test utilities -- do NOT touch the output file, safe to run any time to
# sanity-check the pipeline (e.g. after any edit) before kicking off the real multi-day sweep
def compute_ot(snap, sim, lo, hi, nmax=200, seed=0):
    sd = load_snap_data(snap, sim); a = sd["a"]
    h = select_halos(snap, sim, lo, hi, a, nmax=nmax, seed=seed)
    if h["n"]==0:
        print(f"{snap}/{sim} logM[{lo},{hi}]: no_halos"); return None
    p = get_particles(h, sd, n_min=0)
    r = ot_one_bin(h, p, a, lo, hi)
    print(f"{snap}/{sim} logM[{lo},{hi}]: status={r['status']} n_halos={r['n_halos']} "
          f"n_used_dmo={r['n_used_dmo']} n_used_hyd={r['n_used_hyd']} "
          f"n_empty_dmo={r['n_empty_dmo']} n_empty_hyd={r['n_empty_hyd']} "
          f"x10={r['x10']:.4f} ot_reliable={r['ot_reliable']} "
          f"has dmo_halo_id={'dmo_halo_id' in h} hyd_halo_id={'hyd_halo_id' in h}")
    return r

In [8]:
_ = compute_ot(77, "fgas-4sigma", 13.5, 13.625, nmax=200, seed=0)

77/fgas-4sigma logM[13.5,13.625]: status=ok n_halos=200 n_used_dmo=200 n_used_hyd=200 n_empty_dmo=0 n_empty_hyd=0 x10=0.0825 ot_reliable=True has dmo_halo_id=True hyd_halo_id=True


## What gets saved, per bin (`/{snap}/{sim}/{lo:.3f}_{hi:.3f}/`)

**Attributes** (scalars): `lo, hi` (mass bin edges), `a` (scale factor), `n_halos` (how many
halos this bin actually used, up to NMAX), `status` (`ok`/`all_empty`/`degenerate`/`no_halos`),
`dmo_r200_med`, `x10` (inner-bin edge in units of R200c), `n_empty_dmo/hyd` (stacked radial bins
with zero particles), `n_used_dmo/hyd` (halos that contributed >=1 particle), `npart_dmo/hyd_med`
(median particles/halo), `ot_reliable` (bool -- passes the x10/empty-bin/used-fraction gate),
`n_log, nmax, seed, x10_max` (run config, so any bin is self-describing), and `complete` (the
resume-safety marker).

**Datasets** (arrays): `edges, rc` (radial bin edges/centres, r/R200c), `avg_dmo, avg_hyd`
(stacked normalized mass profiles), `cnt_dmo, cnt_hyd` (raw particle counts per radial bin),
`m_dmo, m_hyd` (per-halo normalized profiles, shape `[n_halos, n_radial_bins]` -- this is what
scales directly with NMAX and dominates file size), `has_dmo, has_hyd` (per-halo: did it
contribute), `npart_dmo, npart_hyd` (per-halo particle counts), `ot` (the OT displacement map,
only present if `status=='ok'`), and now `dmo_halo_id, hyd_halo_id, dmo_m200, hyd_m200, dmo_r200,
hyd_r200` (the exact halo selection that went into this bin).

**Disk cost**: `profiles_v3.hdf5` (NMAX=1000) is 827MB. `m_dmo`/`m_hyd` are the dominant
contributor and scale linearly with `n_halos`, so bins that were already saturating the old
NMAX=1000 cap (common at low mass, where >1000 halos exist) could grow up to ~10x; bins with
fewer than 1000 halos available won't grow at all (NMAX only binds when more halos exist than
the cap allows). Realistic estimate: a few GB, well within your 442GB free -- not a concern.

In [9]:
def save_bin(f, snap, sim, r, h):
    """Write one bin result to /{snap}/{sim}/{lo}_{hi}/. Everything plot/refit needs.

    FIX 1 vs profiles_v3 (RESUME SAFETY): writes g.attrs['complete']=True as the LAST operation.
    profiles_v3's run_sweep decided a bin was "done" just by checking whether the group NAME
    existed -- wrong if the process is killed mid-write, since a half-written group would look
    "done" and get silently skipped forever on resume. Checking this attr instead of just group
    existence is what makes resuming after a crash/restart actually safe over a multi-day run.

    FIX 2 vs profiles_v3: saves dmo_halo_id/hyd_halo_id (now that select_halos returns them) so
    every bin is traceable back to the exact halos used.
    """
    g=f.require_group(f"{snap}/{sim}").create_group(f"{r['lo']:.3f}_{r['hi']:.3f}")
    for k in ("lo","hi","a","n_halos","status","dmo_r200_med","x10",
              "n_empty_dmo","n_empty_hyd","n_used_dmo","n_used_hyd",
              "npart_dmo_med","npart_hyd_med","ot_reliable"):
        g.attrs[k]=r[k]
    if "err" in r: g.attrs["err"]=r["err"]
    g.attrs["n_log"]=N_LOG; g.attrs["nmax"]=NMAX; g.attrs["seed"]=SEED; g.attrs["x10_max"]=X10_MAX
    for k in ("edges","rc","avg_dmo","avg_hyd","cnt_dmo","cnt_hyd",
              "m_dmo","m_hyd","has_dmo","has_hyd","npart_dmo","npart_hyd"):
        g.create_dataset(k,data=r[k],compression="gzip",compression_opts=4)
    if r["ot"] is not None:
        g.create_dataset("ot",data=r["ot"],compression="gzip",compression_opts=4)
    for k in ("dmo_halo_id","hyd_halo_id","dmo_m200","hyd_m200","dmo_r200","hyd_r200"):
        if k in h: g.create_dataset(k,data=h[k],compression="gzip",compression_opts=4)
    g.attrs["complete"]=True             # must be last -- marks this bin as safely resumable
    f.flush()

In [10]:
def run_sweep(out_h5=OUT_H5):
    """Resumable sweep over every (sim, snap, mass bin) -- ~8 sims x 13 snaps x 40 bins = 4160
    bins total. Safe to kill (Ctrl-C, crash, machine restart) and restart at any point: re-checks
    each bin's 'complete' attr (not just whether the group exists) so a bin interrupted mid-write
    gets redone rather than silently treated as finished. One bad bin (bad data, unexpected
    exception) is logged and skipped rather than killing the whole run.
    """
    log=open(out_h5.replace(".hdf5","_log.csv"),"a")
    if log.tell()==0: log.write("snap,sim,lo,hi,status,ot_reliable,x10,n_halos,n_used_dmo,"
        "n_used_hyd,n_empty_dmo,n_empty_hyd,npart_dmo_med,npart_hyd_med,sec\n")
    with h5py.File(out_h5,"a") as f:
        for sim in SIMS:                                            # one sim at a time (RAM)
            for snap in SNAPS:
                gname=f"{snap}/{sim}"
                grp = f[gname] if gname in f else None
                done = {k for k in (grp.keys() if grp else [])
                        if grp[k].attrs.get("complete", False)}
                todo=[(lo,hi) for lo,hi in MASS_BINS if f"{lo:.3f}_{hi:.3f}" not in done]
                if not todo: print(f"{gname}: complete, skipping"); continue
                if grp is not None:            # drop any partial group from an interrupted run
                    for k in list(grp.keys()):
                        if k not in done:
                            print(f"  clearing incomplete bin {gname}/{k} from a previous run")
                            del grp[k]
                try: sd=load_snap_data(snap,sim); a=sd["a"]         # load snapshot ONCE
                except (FileNotFoundError,OSError,KeyError,ValueError) as e:
                    print(f"MISSING {gname}: {e}"); continue
                for lo,hi in todo:
                    t0=time.time()
                    try:
                        h=select_halos(snap,sim,lo,hi,a,nmax=NMAX,seed=SEED)
                        if h["n"]==0:
                            g=f.require_group(gname).create_group(f"{lo:.3f}_{hi:.3f}")
                            g.attrs.update(lo=lo,hi=hi,a=a,n_halos=0,status="no_halos",
                                           ot_reliable=False,complete=True)
                            log.write(f"{snap},{sim},{lo},{hi},no_halos,False,nan,0,0,0,0,0,0,0,"
                                      f"{time.time()-t0:.1f}\n"); log.flush(); continue
                        p=get_particles(h,sd,n_min=0)                # reuses sd's KD-trees
                        r=ot_one_bin(h,p,a,lo,hi)
                        save_bin(f,snap,sim,r,h)
                        msg=(f"{snap}/{sim} [{lo},{hi}] status={r['status']} rel={r['ot_reliable']} "
                             f"x10={r['x10']:.3f} n_halos={r['n_halos']} "
                             f"empty d/h={r['n_empty_dmo']}/{r['n_empty_hyd']} "
                             f"npart d/h={r['npart_dmo_med']:.0f}/{r['npart_hyd_med']:.0f} "
                             f"{time.time()-t0:.1f}s")
                        print(msg)
                        log.write(f"{snap},{sim},{lo},{hi},{r['status']},{r['ot_reliable']},"
                            f"{r['x10']:.4f},{r['n_halos']},{r['n_used_dmo']},{r['n_used_hyd']},"
                            f"{r['n_empty_dmo']},{r['n_empty_hyd']},{r['npart_dmo_med']:.0f},"
                            f"{r['npart_hyd_med']:.0f},{time.time()-t0:.1f}\n"); log.flush()
                    except Exception as e:
                        print(f"ERROR {snap}/{sim} [{lo},{hi}]: {e}")
                        log.write(f"{snap},{sim},{lo},{hi},error,False,nan,0,0,0,0,0,0,0,"
                                  f"{time.time()-t0:.1f}\n"); log.flush()
                del sd                                               # free before next snapshot
    log.close()

def load_bin(out_h5, snap, sim, lo, hi):
    """Read one saved bin back out as a dict (attrs + datasets)."""
    with h5py.File(out_h5,"r") as f:
        g=f[f"{snap}/{sim}/{lo:.3f}_{hi:.3f}"]
        r={**{k:g.attrs[k] for k in g.attrs},**{k:g[k][...] for k in g.keys()}}
    r["snap"],r["sim"]=snap,sim
    return r

## Launch the real sweep

Not run yet -- uncomment when ready. Expect roughly a week; safe to interrupt and rerun this
exact cell at any point, it resumes from the last completed bin (see the resume-safety note at
the top).

## Gap-fill: Mstar-1sigma / snap 69

The main sweep hit one real data problem: `flamingo_downsampled/Mstar-1sigma/flamingo_0069.hdf5`'s `BHParticles` block is missing `DynamicalMasses` (only `Coordinates` survived -- confirmed against snap 67 of the same sim, which has all 4 fields normally). This is a defect in that one source file, not something this pipeline caused or can regenerate. Fixed by reprocessing just that snap with BH particles excluded (DM+gas+stars intact), writing into the same output file and log, with an explicit `bh_excluded=True` attr on those 40 bins so the difference is always traceable, never silently ambiguous.

In [11]:
# one-off gap-fill: Mstar-1sigma / snap 69's BHParticles block in the SOURCE file is missing
# DynamicalMasses/SubgridMasses/Velocities (only 'Coordinates' survived -- confirmed by comparing
# to snap 67 of the same sim, which has all 4 fields normally). This is a pre-existing defect in
# that one downsampled file, not something this pipeline caused or can regenerate. _read_tolerant
# skips a particle type if its mass field is missing (with a loud warning) instead of raising --
# used ONLY here, NOT swapped into the main run_sweep, so a missing field anywhere else still
# fails loudly instead of being silently masked.
def _read_tolerant(path, parts):
    pos, ms = [], []
    with h5py.File(path, "r") as f:
        for p in parts:
            if p not in f or "Coordinates" not in f[p]: continue
            c = f[p]["Coordinates"][:]
            if c.shape[0]==0: continue
            mk = _PARTS[p]
            if mk not in f[p]:
                print(f"  WARNING: {p}/{mk} missing in {path} -- excluding this particle type "
                      f"(fields present: {list(f[p])})")
                continue
            pos.append(c); ms.append(f[p][mk][:])
    if not pos: raise ValueError(f"no particles in {path}")
    return np.concatenate(pos)%BOX_MPC, np.concatenate(ms)

def load_snap_data_tolerant(snap, sim):
    dp,dm = _read(f"{DMO_DIR}/{FNAME.format(snap=snap)}", ["DMParticles"])
    hp,hm = _read_tolerant(f"{HYD_DIR.format(sim=sim)}/{FNAME.format(snap=snap)}", list(_PARTS))
    return {"dmo_pos":dp,"dmo_m":dm,"dmo_tree":cKDTree(dp,boxsize=BOX_MPC),
            "hyd_pos":hp,"hyd_m":hm,"hyd_tree":cKDTree(hp,boxsize=BOX_MPC),
            "a":read_a(snap,sim)}

In [12]:
# fill the one known gap: Mstar-1sigma / snap 69, BH particles excluded for this snap only.
# Writes into the SAME output file + log as the main sweep, using the exact same save_bin/
# ot_one_bin/select_halos/get_particles -- these 40 bins slot in identically to every other bin,
# just with an extra 'bh_excluded' attr recording the one difference.
GAP_SNAP, GAP_SIM = 69, "Mstar-1sigma"

sd = load_snap_data_tolerant(GAP_SNAP, GAP_SIM)
a = sd["a"]

log = open(OUT_H5.replace(".hdf5","_log.csv"), "a")
with h5py.File(OUT_H5, "a") as f:
    gname = f"{GAP_SNAP}/{GAP_SIM}"
    done = {k for k in (f[gname].keys() if gname in f else [])
            if f[gname][k].attrs.get("complete", False)}
    todo = [(lo,hi) for lo,hi in MASS_BINS if f"{lo:.3f}_{hi:.3f}" not in done]
    print(f"{gname}: {len(done)} already done, {len(todo)} to fill")
    for lo, hi in todo:
        t0 = time.time()
        try:
            h = select_halos(GAP_SNAP, GAP_SIM, lo, hi, a, nmax=NMAX, seed=SEED)
            if h["n"]==0:
                g=f.require_group(gname).create_group(f"{lo:.3f}_{hi:.3f}")
                g.attrs.update(lo=lo,hi=hi,a=a,n_halos=0,status="no_halos",
                               ot_reliable=False,bh_excluded=True,complete=True)
                log.write(f"{GAP_SNAP},{GAP_SIM},{lo},{hi},no_halos,False,nan,0,0,0,0,0,0,0,"
                          f"{time.time()-t0:.1f}\n"); log.flush(); continue
            p = get_particles(h, sd, n_min=0)
            r = ot_one_bin(h, p, a, lo, hi)
            save_bin(f, GAP_SNAP, GAP_SIM, r, h)
            f[f"{gname}/{lo:.3f}_{hi:.3f}"].attrs["bh_excluded"] = True   # document the one difference
            print(f"{GAP_SNAP}/{GAP_SIM} [{lo},{hi}] status={r['status']} rel={r['ot_reliable']} "
                  f"n_halos={r['n_halos']} {time.time()-t0:.1f}s")
            log.write(f"{GAP_SNAP},{GAP_SIM},{lo},{hi},{r['status']},{r['ot_reliable']},"
                f"{r['x10']:.4f},{r['n_halos']},{r['n_used_dmo']},{r['n_used_hyd']},"
                f"{r['n_empty_dmo']},{r['n_empty_hyd']},{r['npart_dmo_med']:.0f},"
                f"{r['npart_hyd_med']:.0f},{time.time()-t0:.1f}\n"); log.flush()
        except Exception as e:
            print(f"ERROR {GAP_SNAP}/{GAP_SIM} [{lo},{hi}]: {e}")
            log.write(f"{GAP_SNAP},{GAP_SIM},{lo},{hi},error,False,nan,0,0,0,0,0,0,0,"
                      f"{time.time()-t0:.1f}\n"); log.flush()
del sd
log.close()
print("Mstar-1sigma / snap 69 gap-fill complete")

69/Mstar-1sigma: 0 already done, 40 to fill


69/Mstar-1sigma [10.0,10.125] status=ok rel=False n_halos=48 0.3s


69/Mstar-1sigma [10.125,10.25] status=ok rel=False n_halos=29 0.2s


69/Mstar-1sigma [10.25,10.375] status=ok rel=False n_halos=459 1.1s


69/Mstar-1sigma [10.375,10.5] status=ok rel=False n_halos=1236 1.5s


...

69/Mstar-1sigma [14.375,14.5] status=ok rel=True n_halos=394 1.1s


69/Mstar-1sigma [14.5,14.625] status=ok rel=True n_halos=180 0.6s


69/Mstar-1sigma [14.625,14.75] status=ok rel=True n_halos=94 0.4s


69/Mstar-1sigma [14.75,14.875] status=ok rel=True n_halos=35 0.3s


69/Mstar-1sigma [14.875,15.0] status=ok rel=True n_halos=15 0.2s
Mstar-1sigma / snap 69 gap-fill complete


In [ ]:
# run_sweep(OUT_H5)   # <-- uncomment to start the full sweep (8 sims x 13 snaps x 40 bins)